# Bounded selectors and ES256

Use range selectors under Ed25519 major 2 and P-256 major 3, then check
canonical numeric refusals on every wire major.

Install Deno and register its TypeScript kernel with `deno jupyter --install`.
Open this file in a Jupyter front end such as JupyterLab or VS Code, select
**Deno**, then Run All in a fresh kernel. The cells import exact published
signer **2.0.0** and verifier **2.0.0**.

Fixed times describe demonstration credentials. Generated keys, signatures and
nonces stay in memory. Signing and verification do not grant operational
authority; the host owns trust, live revocation, replay and policy.

## Load the published packages

Check both release identities and the backend Deno selects.


In [ ]:
import assert from "node:assert/strict";
import * as signer from "npm:@bounded-authority-protocol/signer@2.0.0";
import * as verifier from "npm:@bounded-authority-protocol/verifier@2.0.0";
import signerMetadata from "npm:@bounded-authority-protocol/signer@2.0.0/package.json" with {
  type: "json",
};
import verifierMetadata from "npm:@bounded-authority-protocol/verifier@2.0.0/package.json" with {
  type: "json",
};
assert.equal(signerMetadata.version, "2.0.0");
assert.equal(verifierMetadata.version, "2.0.0");
assert.equal(verifier.BACKEND_ID, "portable");
console.log(
  JSON.stringify({
    signer: signerMetadata.version,
    verifier: verifierMetadata.version,
    backend: verifier.BACKEND_ID,
  }),
);

{"signer":"2.0.0","verifier":"2.0.0","backend":"portable"}


## Prepare real demonstration key handles

Each constructor generates a fresh non-extractable WebCrypto key pair and wraps
it with the signer's own handle factory, so the private key never leaves
WebCrypto. No key material is displayed. Production custody supplies this same
key-handle contract.

In [ ]:
async function createEd25519Handle(
  keyId: string,
  role: signer.KeyRole = "holder",
): Promise<signer.KeyHandle> {
  const pair = await crypto.subtle.generateKey({ name: "Ed25519" }, false, [
    "sign",
    "verify",
  ]) as CryptoKeyPair;
  const made = await signer.ed25519KeyHandle(pair, { keyId, role });
  if (!made.ok) throw new Error(made.error);
  return made.value;
}
async function createEs256Handle(
  keyId: string,
  role: signer.KeyRole = "holder",
): Promise<signer.Es256KeyHandle> {
  const pair = await crypto.subtle.generateKey(
    { name: "ECDSA", namedCurve: "P-256" },
    false,
    ["sign", "verify"],
  ) as CryptoKeyPair;
  const made = await signer.es256KeyHandle(pair, { keyId, role });
  if (!made.ok) throw new Error(made.error);
  return made.value;
}
const ed25519Width =
  (await (await createEd25519Handle("example-ed25519")).publicKey()).length;
const es256Width =
  (await (await createEs256Handle("example-es256")).publicKey()).length;
assert.equal(ed25519Width, 32);
assert.equal(es256Width, 65);
console.log(
  JSON.stringify({ ed25519Bytes: ed25519Width, es256Bytes: es256Width }),
);

{"ed25519Bytes":32,"es256Bytes":65}


## Issue a major 2 range grant

The grant bounds the temperature between two fractional tagged values.


In [ ]:
const issuer2 = await createEd25519Handle("issuer-2", "issuer");
const holder2 = await createEd25519Handle("holder-2");
const rangeGrant2 = await signer.signV2Grant({
  issuer: "https://climate.example",
  grantId: "urn:example:grant:temperature-2",
  audiences: ["https://thermostat.example"],
  issuedAt: 1_000,
  notBefore: 1_000,
  expiresAt: 2_000,
  holderThumbprint: await holder2.thumbprint(),
  operations: [{
    name: "set_temperature",
    selectors: [
      { kind: "gte", path: ["celsius"], value: { t: "float", v: 18.5 } },
      { kind: "lte", path: ["celsius"], value: { t: "float", v: 24.5 } },
    ],
  }],
}, issuer2);
assert.ok(rangeGrant2.ok);
async function verifyTemperature2(celsius: number) {
  const request = {
    method: "PUT",
    targetUri: "https://thermostat.example/setpoint",
    invocationId: "3b9e1c52-8d4f-4a1e-b7c2-5e6f7a8b9c0d",
    operation: "set_temperature",
    castArguments: {
      t: "object",
      v: new Map<string, verifier.Tagged>([["celsius", {
        t: "float",
        v: celsius,
      }]]),
    } satisfies verifier.Tagged,
  };
  const envelope = await signer.signV2Report(
    { ...request, grantCompact: rangeGrant2.value.grant },
    holder2,
    { issuedAt: 1_100, proofId: "temperature-proof" },
  );
  assert.ok(envelope.ok);
  return verifier.v2.checkEnvelope(envelope.value.grant, envelope.value.proof, {
    ...request,
    trustedIssuer: { keyId: "issuer-2", publicKey: await issuer2.publicKey() },
    issuer: "https://climate.example",
    audience: "https://thermostat.example",
    evaluationTime: 1_120,
    clockSkew: 0,
    proofMaxAge: 300,
    nonce: { kind: "not_required" },
  });
}
console.log(JSON.stringify({ ok: rangeGrant2.ok }));

{"ok":true}


## Verify an in-range major 2 request

The displayed operation comes from the verified facts.


In [ ]:
const within2 = await verifyTemperature2(21.5);
assert.ok(within2.ok);
console.log(
  JSON.stringify({ ok: within2.ok, operation: within2.value.operation }),
);

{"ok":true,"operation":"set_temperature"}


## Refuse an out-of-range major 2 request

The holder signs the out-of-range request successfully; the selector check must
refuse it.

In [ ]:
const outside2 = await verifyTemperature2(30.5);
assert.deepEqual(outside2, { ok: false });
console.log(JSON.stringify(outside2));


{"ok":false}


## Issue a major 3 range grant

The grant bounds the temperature between two fractional tagged values.


In [ ]:
const issuer3 = await createEs256Handle("issuer-3", "issuer");
const holder3 = await createEs256Handle("holder-3");
const rangeGrant3 = await signer.signV3Grant({
  issuer: "https://climate.example",
  grantId: "urn:example:grant:temperature-3",
  audiences: ["https://thermostat.example"],
  issuedAt: 1_000,
  notBefore: 1_000,
  expiresAt: 2_000,
  holderThumbprint: await holder3.thumbprint(),
  operations: [{
    name: "set_temperature",
    selectors: [
      { kind: "gte", path: ["celsius"], value: { t: "float", v: 18.5 } },
      { kind: "lte", path: ["celsius"], value: { t: "float", v: 24.5 } },
    ],
  }],
}, issuer3);
assert.ok(rangeGrant3.ok);
async function verifyTemperature3(celsius: number) {
  const request = {
    method: "PUT",
    targetUri: "https://thermostat.example/setpoint",
    invocationId: "3b9e1c52-8d4f-4a1e-b7c2-5e6f7a8b9c0d",
    operation: "set_temperature",
    castArguments: {
      t: "object",
      v: new Map<string, verifier.Tagged>([["celsius", {
        t: "float",
        v: celsius,
      }]]),
    } satisfies verifier.Tagged,
  };
  const envelope = await signer.signV3Report(
    { ...request, grantCompact: rangeGrant3.value.grant },
    holder3,
    { issuedAt: 1_100, proofId: "temperature-proof" },
  );
  assert.ok(envelope.ok);
  return verifier.v3.checkEnvelope(envelope.value.grant, envelope.value.proof, {
    ...request,
    trustedIssuer: { keyId: "issuer-3", publicKey: await issuer3.publicKey() },
    issuer: "https://climate.example",
    audience: "https://thermostat.example",
    evaluationTime: 1_120,
    clockSkew: 0,
    proofMaxAge: 300,
    nonce: { kind: "not_required" },
  });
}
console.log(JSON.stringify({ ok: rangeGrant3.ok }));

{"ok":true}


## Verify an in-range major 3 request

The displayed operation comes from the verified facts.


In [ ]:
const within3 = await verifyTemperature3(21.5);
assert.ok(within3.ok);
console.log(
  JSON.stringify({ ok: within3.ok, operation: within3.value.operation }),
);

{"ok":true,"operation":"set_temperature"}


## Refuse an out-of-range major 3 request

The holder signs the out-of-range request successfully; the selector check must
refuse it.

In [ ]:
const outside3 = await verifyTemperature3(30.5);
assert.deepEqual(outside3, { ok: false });
console.log(JSON.stringify(outside3));


{"ok":false}


## Refuse an integral float on major 1

Use an integer tag for whole numbers; an integral value under a float tag is
noncanonical.

In [ ]:
const integral1 = await signer.signGrant({
  issuer: "https://climate.example",
  grantId: "urn:example:grant:integral-1",
  audiences: ["https://thermostat.example"],
  issuedAt: 1_000,
  notBefore: 1_000,
  expiresAt: 2_000,
  holderThumbprint: await holder2.thumbprint(),
  operations: [{
    name: "set_retention",
    selectors: [{
      kind: "equals",
      path: ["days"],
      value: { t: "float", v: 30.0 },
    }],
  }],
}, issuer2);
assert.equal(integral1.ok, false);
if (integral1.ok) throw new Error("integral float accepted");
assert.equal(integral1.error, "producer_error");
console.log(JSON.stringify({ ok: integral1.ok, code: integral1.error }));

{"ok":false,"code":"producer_error"}


## Refuse an integral float on major 2

Use an integer tag for whole numbers; an integral value under a float tag is
noncanonical.

In [ ]:
const integral2 = await signer.signV2Grant({
  issuer: "https://climate.example",
  grantId: "urn:example:grant:integral-2",
  audiences: ["https://thermostat.example"],
  issuedAt: 1_000,
  notBefore: 1_000,
  expiresAt: 2_000,
  holderThumbprint: await holder2.thumbprint(),
  operations: [{
    name: "set_retention",
    selectors: [{
      kind: "lte",
      path: ["days"],
      value: { t: "float", v: 30.0 },
    }],
  }],
}, issuer2);
assert.equal(integral2.ok, false);
if (integral2.ok) throw new Error("integral float accepted");
assert.equal(integral2.error, "producer_error");
console.log(JSON.stringify({ ok: integral2.ok, code: integral2.error }));

{"ok":false,"code":"producer_error"}


## Refuse an integral float on major 3

Use an integer tag for whole numbers; an integral value under a float tag is
noncanonical.

In [ ]:
const integral3 = await signer.signV3Grant({
  issuer: "https://climate.example",
  grantId: "urn:example:grant:integral-3",
  audiences: ["https://thermostat.example"],
  issuedAt: 1_000,
  notBefore: 1_000,
  expiresAt: 2_000,
  holderThumbprint: await holder3.thumbprint(),
  operations: [{
    name: "set_retention",
    selectors: [{
      kind: "lte",
      path: ["days"],
      value: { t: "float", v: 30.0 },
    }],
  }],
}, issuer3);
assert.equal(integral3.ok, false);
if (integral3.ok) throw new Error("integral float accepted");
assert.equal(integral3.error, "producer_error");
console.log(JSON.stringify({ ok: integral3.ok, code: integral3.error }));

{"ok":false,"code":"producer_error"}


## Sign the canonical integer replacement

The same selector works when its tagged numeric value is canonical.


In [ ]:
const integerGrant = await signer.signV3Grant({
  issuer: "https://climate.example",
  grantId: "urn:example:grant:integer",
  audiences: ["https://thermostat.example"],
  issuedAt: 1_000,
  notBefore: 1_000,
  expiresAt: 2_000,
  holderThumbprint: await holder3.thumbprint(),
  operations: [{
    name: "set_retention",
    selectors: [{ kind: "lte", path: ["days"], value: { t: "int", v: 30 } }],
  }],
}, issuer3);
assert.ok(integerGrant.ok);
console.log(JSON.stringify({ ok: integerGrant.ok }));

{"ok":true}


## Sign the canonical fraction replacement

The same selector works when its tagged numeric value is canonical.


In [ ]:
const fractionGrant = await signer.signV3Grant({
  issuer: "https://climate.example",
  grantId: "urn:example:grant:fraction",
  audiences: ["https://thermostat.example"],
  issuedAt: 1_000,
  notBefore: 1_000,
  expiresAt: 2_000,
  holderThumbprint: await holder3.thumbprint(),
  operations: [{
    name: "set_retention",
    selectors: [{
      kind: "lte",
      path: ["days"],
      value: { t: "float", v: 30.5 },
    }],
  }],
}, issuer3);
assert.ok(fractionGrant.ok);
console.log(JSON.stringify({ ok: fractionGrant.ok }));

{"ok":true}


## Finish

Shut down the kernel to discard the demonstration keys. No private material was
persisted. Continue with the
[getting started guide](../guides/getting-started.md).